# Preprocess of IMDB data to create an IMDB content dataset

## Imports

In [30]:
import numpy as np
import pandas as pd
import zipfile as zipfile

import sys
import requests
import os

In [31]:
# URL of datasources
url_mdl                = 'https://files.grouplens.org/datasets/movielens/ml-20m.zip'

url_imdb_namebasics    = 'https://datasets.imdbws.com/name.basics.tsv.gz'
url_imdb_titleakas     = 'https://datasets.imdbws.com/title.akas.tsv.gz'
url_imdb_titlecrew     = 'https://datasets.imdbws.com/title.crew.tsv.gz'
url_imdb_titlebasics   = 'https://datasets.imdbws.com/title.basics.tsv.gz'
url_imdb_titleepisode  = 'https://datasets.imdbws.com/title.episode.tsv.gz'
url_imdb_titleprincipals  = 'https://datasets.imdbws.com/title.principals.tsv.gz'
url_imdb_titleratings  = 'https://datasets.imdbws.com/title.ratings.tsv.gz'

## Functions

## Preprocess

In [32]:
# Load : A ajouter : chargement direct depuis url
title_crew = pd.read_csv('data_imdb/title.crew.tsv.gz',compression='gzip', sep='\t', na_values=['\\N', 'nan']) # note : \\N is required to avoid confusion with newline '\n' 

# preprocess of title_crew.tconst to match imdbId from mdl
title_crew['tconst_short'] = [x[-7:] for x in title_crew['tconst']]
title_crew['tconst_short'] = title_crew['tconst_short'].astype('int64')

# Limitation of the data set
list_movies = np.genfromtxt('data_processed/list_imdbId_2000_2010.csv',delimiter=",", dtype=int)
title_crew = title_crew[title_crew['tconst_short'].isin(list_movies)]

# Gestions de NANs
title_crew = title_crew.fillna('')
#title_crew = title_crew.dropna(axis=0, how='any', subset=['directors'])

title_crew.head(5)
#title_crew.isna().sum()

,tconst,directors,writers,tconst_short
34803,tt0035423,nm0003506,"nm0737216,nm0003506",35423
93938,tt0096056,nm0324875,"nm0234502,nm0324875",96056
115402,tt0118141,nm0000417,nm0000417,118141
115828,tt0118589,nm0193554,"nm0921985,nm0486824",118589
116068,tt0118840,nm0885666,nm0885666,118840


In [33]:
# Load Name
name_basics = pd.read_csv('data_imdb/name.basics.tsv.gz',compression='gzip', sep='\t', usecols=['nconst', 'primaryName'])

print(name_basics.info())
name_basics.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12354442 entries, 0 to 12354441
Data columns (total 2 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   nconst       object
 1   primaryName  object
dtypes: object(2)
memory usage: 188.5+ MB
None


,nconst,primaryName
0,nm0000001,Fred Astaire
1,nm0000002,Lauren Bacall
2,nm0000003,Brigitte Bardot
3,nm0000004,John Belushi
4,nm0000005,Ingmar Bergman


In [34]:
# Merge to add names
title_crew = title_crew.merge(right=name_basics, left_on='directors', right_on='nconst', how='left')
title_crew = title_crew.rename({'primaryName':'directorsName'})
title_crew.head(5)

,tconst,directors,writers,tconst_short,nconst,primaryName
0,tt0035423,nm0003506,"nm0737216,nm0003506",35423,nm0003506,James Mangold
1,tt0096056,nm0324875,"nm0234502,nm0324875",96056,nm0324875,Menahem Golan
2,tt0118141,nm0000417,nm0000417,118141,nm0000417,Crispin Glover
3,tt0118589,nm0193554,"nm0921985,nm0486824",118589,nm0193554,Vondie Curtis-Hall
4,tt0118840,nm0885666,nm0885666,118840,nm0885666,Jim Van Bebber


In [35]:
# Aggregation of directors in list
title_crew = title_crew.groupby('tconst_short')['primaryName'].apply(list)
title_crew = pd.DataFrame(title_crew).reset_index()

# Transform list => string
title_crew['primaryName_str'] = [' '.join(map(str, l)) for l in title_crew['primaryName']]

print(title_crew.info())
title_crew.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7256 entries, 0 to 7255
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   tconst_short     7256 non-null   int64 
 1   primaryName      7256 non-null   object
 2   primaryName_str  7256 non-null   object
dtypes: int64(1), object(2)
memory usage: 170.2+ KB
None


,tconst_short,primaryName,primaryName_str
0,35423,[James Mangold],James Mangold
1,96056,"[Menahem Golan, nan]",Menahem Golan nan
2,118141,[Crispin Glover],Crispin Glover
3,118589,[Vondie Curtis-Hall],Vondie Curtis-Hall
4,118840,"[Jim Van Bebber, nan, nan]",Jim Van Bebber nan nan


## Merge

In [37]:
imdb_content = title_crew


imdb_content.head(5)

,tconst_short,primaryName,primaryName_str
0,35423,[James Mangold],James Mangold
1,96056,"[Menahem Golan, nan]",Menahem Golan nan
2,118141,[Crispin Glover],Crispin Glover
3,118589,[Vondie Curtis-Hall],Vondie Curtis-Hall
4,118840,"[Jim Van Bebber, nan, nan]",Jim Van Bebber nan nan


## Save file

In [38]:
imdb_content.to_csv("data_processed/imdb_content.csv.zip", index=False, compression="zip")